# Tutorial: Simple Recent Memory Walkthrough

Audience:
- Engineers who want the low-dependency backend first.

Prerequisites:
- Basic Python familiarity.
- Understanding of recent-history style memory.

Learning goals:
- Build a `simple_recent` engine.
- Record a comment and a vote.
- Inspect deterministic relationship and browse context outputs.


## Outline

1. Setup the import path and runtime stub
2. Create a `simple_recent` engine
3. Record recent interactions
4. Inspect reply, relationship, and browse state


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from yclient_memory import build_memory_engine
from yclient_memory.config import MemoryConfig
from yclient_memory.contracts import BrowseMemoryRequest, CommentMemoryEvent, RelationshipSignalRequest, ReplyMemoryRequest, VoteMemoryEvent

class RuntimeStub:
    def llm_json(self, prompt_key, variables, config=None):
        return {}

    def llm_text(self, prompt_key, variables, config=None):
        return ''

    def get_author_id_and_username(self, post_id):
        return 55, 'alice'

    def get_thread_root_id(self, post_id):
        return 900

    def get_recent_root_posts(self, round_id, limit=24, rounds_back=18):
        return []

    def get_post_text(self, post_id):
        return 'that benchmark evidence is weak'

    def persona_snapshot(self):
        return {}

    def decision_log(self, payload):
        pass

runtime = RuntimeStub()
config = MemoryConfig.from_mapping({'memory_backend': 'simple_recent'})
engine = build_memory_engine(backend='simple_recent', config=config, runtime=runtime)
engine.capabilities()


## Step 1 - Record lightweight relationship history

`simple_recent` does not synthesize digests or reflections. It keeps recent pair and thread history plus a lightweight relationship state.


In [ ]:
engine.record_comment(
    CommentMemoryEvent(
        round_id=1,
        target_post_id=10,
        thread_root_id=900,
        other_user_id=55,
        other_username='alice',
        other_text='that benchmark is weak',
        my_text='nah your benchmark evidence is weak',
        conv_text='thread text',
    )
)
engine.record_vote(VoteMemoryEvent(round_id=2, post_id=10, vote_type='like'))
reply = engine.build_reply_context(
    ReplyMemoryRequest(query_text='compose a reply', other_user_id=55, other_username='alice', thread_root_id=900)
)
relationship = engine.relationship_signal(RelationshipSignalRequest(other_user_id=55))
browse = engine.build_browse_context(BrowseMemoryRequest(thread_root_id=900, round_id=2))
{
    'reply_context': reply.rendered_text,
    'relationship': relationship,
    'browse_context': browse.rendered_text,
}


## Exercises

- Set `vote_signal_only=False` and compare the reply context.
- Call `maintenance_tick()` with a later round to see relationship decay.
- Record interactions with a second user and compare relationship summaries.


In [ ]:
engine.maintenance_tick(type('Req', (), {'round_id': 5})())
engine.relationship_signal(RelationshipSignalRequest(other_user_id=55))
